# SML Fine-tuning on Kaggle (2× T4)

LoRA fine-tuning of `Qwen/Qwen3.5-4B` on internal document QA pairs using [ms-swift](https://swift.readthedocs.io/en/latest/GetStarted/SWIFT-installation.html).  
Training runs on 2× NVIDIA T4 GPUs (Kaggle free tier) with `CUDA_VISIBLE_DEVICES=0,1`.

## 1. Install Dependencies

In [1]:
!pip install uv
!uv pip install 'ms-swift' -U
!uv pip install wandb
!uv pip install -U \
  "transformers>=4.56.2" \
  "peft==0.11.1" \
  "qwen_vl_utils>=0.0.14"
!uv pip install hf

Using Python 3.12.12 environment at: /usr
Resolved 147 packages in 1.00s                                       
Prepared 7 packages in 456ms                                             
Uninstalled 7 packages in 589ms
Installed 7 packages in 285ms                               
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - fsspec==2026.3.0
 + fsspec==2025.3.0
 - peft==0.11.1
 + peft==0.19.1
 - pillow==12.2.0
 + pillow==11.3.0
 - protobuf==6.33.6
 + protobuf==7.34.1
 - torch==2.10.0
 + torch==2.11.0
 - transformers==5.7.0
 + transformers==5.6.2
Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 368ms                                        
Uninstalled 1 package in 1ms
Installed 1 package in 2ms                                  
 - protobuf==7.34.1
 + protobuf==6.33.6
Using Python 3.12.12 environment at: /usr
Resolved 62 packages in 447ms                                        
Prepared 4 packages in 1ms                                               
Uninstalled 4 p

Install compatible torch (ms-swift is sensitive to torch + CUDA version combinations).

In [2]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0
!pip uninstall -y cuml-cu12 cudf-cu12 libcuvs-cu12 libraft-cu12

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
  Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl (915.6 MB)
Using cached torchvision-0.25.0-cp312-cp312-manylinux_2_28_x86_64.whl (8.1 MB)
Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.9 MB)
Using cached cuda_bindings-1

## 2. Verify Installation

Checks Python, CUDA, and key package versions.

In [4]:
import sys
import importlib
from packaging import version

def check(pkg_name, min_ver=None, max_ver=None):
    try:
        module = importlib.import_module(pkg_name)
        ver = getattr(module, "__version__", "unknown")
        
        ok = True
        if min_ver and version.parse(ver) < version.parse(min_ver):
            ok = False
        if max_ver and version.parse(ver) >= version.parse(max_ver):
            ok = False

        status = "✅ OK" if ok else "❌ NOT OK"
        print(f"{pkg_name:15} {ver:15} {status}")
    except ImportError:
        print(f"{pkg_name:15} NOT INSTALLED ❌")

# Python
py_ver = sys.version.split()[0]
print(f"{'python':15} {py_ver:15} {'✅ OK' if version.parse(py_ver)>=version.parse('3.10') else '❌ NOT OK'}")

# CUDA (via torch)
try:
    import torch
    cuda_ver = torch.version.cuda
    print(f"{'cuda':15} {cuda_ver:15} (from torch)")
except:
    print(f"{'cuda':15} NOT AVAILABLE")

print("\n--- Package Checks ---")

check("torch", "2.0")
check("transformers", "4.33")
check("modelscope", "1.23")
check("peft", "0.11", "0.20")
check("flash_attn")  # version varies, just check install
check("trl", "0.15", "0.30")
check("deepspeed", "0.14")
check("vllm", "0.5.1")

python          3.12.12         ✅ OK
cuda            12.8            (from torch)

--- Package Checks ---
torch           2.10.0+cu128    ✅ OK
transformers    5.7.0           ✅ OK
modelscope      1.36.3          ✅ OK
peft            0.11.1          ✅ OK
flash_attn      NOT INSTALLED ❌
trl             0.29.1          ✅ OK
deepspeed       NOT INSTALLED ❌
vllm            NOT INSTALLED ❌


## 3. Init Wandb 
Logs training metrics to Wandb

In [ ]:
import wandb
wandb.login()
run = wandb.init(project="sml_training")

## 4. LoRA Fine-tuning

Runs supervised fine-tuning with LoRA on `Qwen/Qwen3.5-4B`. Key configuration:

| Parameter | Value |
|---|---|
| LoRA rank / alpha | 32 / 64 |
| Target modules | `q_proj k_proj v_proj o_proj gate_proj up_proj down_proj` |
| Batch size | 4 × 4 gradient accumulation = effective batch 16 |
| Max sequence length | 512 |
| Learning rate | 2e-4 with 5% warmup |
| Epochs | 6 |
| Attention | `sdpa` (flash_attention unavailable on T4) |
| Eval / save every | 30 steps |

In [5]:
 !CUDA_VISIBLE_DEVICES=0,1 swift sft \
    --model Qwen/Qwen3.5-4B \
    --use_hf true \
    --dataset /kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/merged_qa.jsonl \
    --tuner_type lora \
    --lora_rank 32 \
    --lora_alpha 64 \
    --lora_dropout 0.05 \
    --target_modules q_proj k_proj v_proj,o_proj gate_proj up_proj down_proj \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --max_length 512 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --max_epochs 6 \
    --attn_impl sdpa \
    --output_dir /kaggle/working/sml_training/qwen35 \
    --save_strategy steps \
    --early_stop_interval 3 \
    --eval_steps 30 \
    --save_steps 30 \
    --dataloader_num_workers 2 \
    --dataset_num_proc 8 \
    --load_from_cache_file true \
    --model_author swift \
    --model_name swift-robot \
    --loss_scale default \
    --report_to wandb

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model Qwen/Qwen3.5-4B --use_hf true --dataset /kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/merged_qa.jsonl --tuner_type lora --lora_rank 32 --lora_alpha 64 --lora_dropout 0.05 --target_modules q_proj k_proj v_proj,o_proj gate_proj up_proj down_proj --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --max_length 512 --learning_rate 2e-4 --warmup_ratio 0.05 --max_epochs 6 --attn_impl sdpa --output_dir /kaggle/working/sml_training/qwen35 --save_strategy steps --early_stop_interval 3 --eval_steps 30 --save_steps 30 --dataloader_num_workers 2 --dataset_num_proc 8 --load_from_cache_file true --model_author swift --model_name swift-robot --loss_scale default --report_to wandb`
[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloa

## 5. Merge LoRA Adapter

Merges the trained LoRA weights into the base model and saves the full model weights to `checkpoint-135-merged`.

In [6]:
!swift export --adapters /kaggle/working/sml_training/qwen35/v2-20260429-064312/checkpoint-135 --merge_lora true --use_hf true

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/export.py --adapters /kaggle/working/sml_training/qwen35/v2-20260429-064312/checkpoint-135 --merge_lora true --use_hf true`
[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] Successfully loaded /kaggle/working/sml_training/qwen35/v2-20260429-064312/checkpoint-135/args.json.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-4B
Fetching 12 files: 100%|████████████████████| 12/12 [00:00<00:00, 137518.16it/s]
Download complete: : 0.00B [00:00, ?B/s]              [INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
Download complete: : 0.00B [00:00, ?B/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Pl

## 6. Publish to HuggingFace Hub

Logs in, creates the model repository, and uploads the merged checkpoint.

In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
!hf repo create tnnanh1005/Qwen3.5-4B-SML --repo-type model

✓ Repo created
  repo_id: tnnanh1005/Qwen3.5-4B-SML
  url: https://huggingface.co/tnnanh1005/Qwen3.5-4B-SML


In [9]:
!hf upload tnnanh1005/Qwen3.5-4B-SML /kaggle/working/sml_training/qwen35/v2-20260429-064312/checkpoint-135-merged --repo-type model

Start hashing 11 files.
Finished hashing 11 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...135-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            

Processing Files (1 / 1)      :   0%|              | 20.0MB / 9.10GB, 20.0MB/s  

  ...135-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            

  ...135-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


  ...0002-of-00002.safetensors:   0%|              |  600kB / 4.11GB            

  ...135-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


Processing Files (1 / 2)      :   0%|              | 20.6MB / 9.10GB, 12.9MB/s  
New Data Upload               :   0%|              |  600kB /  134MB,  375kB/s  

  ...135-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


  ...0002-of-00002.safetensors:   0%|         